# 🌍 Earth Field Analysis
## Layer 7 – Earth Field State Engine
| Layer | Name | Status |
|-------|------|--------|
| 0–6 | Data layers | ✅ complete |
| **7** | **Earth Field State Engine** | **← this layer** |
| 8 | Research / Hypotheses | ⬜ |
> **Layer 7 is NOT an analysis layer.** Layer 7 is an **Engine**:
> It takes all outputs from Layer 0–6 and produces a **comparable, time-based, machine-readable Earth Field system state** for Layer 8.
**Engine functions:**
1. Collects all `layer{0..6}_test_state.json`
2. Normalizes scores, levels, confidence
3. Calculates trends (Δ1h, Δ24h, volatility) from history
4. Calculates layer couplings (strength, lag, confidence)
5. Determines dominant / secondary / quiet layers
6. Classifies system state (8 classes)
7. Generates event tags
8. **Appends snapshot to `layer7_test_history.jsonl`** (archive for Layer 8)
9. Saves current state as `layer7_test_state.json`

In [1]:
import warnings; warnings.filterwarnings('ignore')
import datetime, json, math, os
from pathlib import Path
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import pandas as pd
import numpy as np

RUN_TIMESTAMP = datetime.datetime.utcnow().isoformat() + 'Z'
print(f'Engine run: {RUN_TIMESTAMP}')

HISTORY_FILE = 'layer7_test_history.jsonl'
STATE_FILE   = 'layer7_test_state.json'

# Thresholds
LEVEL_THRESHOLDS = {'quiet': 0.30, 'moderate': 0.60, 'active': 0.80}

def classify_level(score):
    if score is None: return 'unknown'
    if score < LEVEL_THRESHOLDS['quiet']:    return 'quiet'
    if score < LEVEL_THRESHOLDS['moderate']: return 'moderate'
    if score < LEVEL_THRESHOLDS['active']:   return 'active'
    return 'strong'

Engine run: 2026-05-12T07:27:09.478526Z


---
## 1.Collect all layer states

In [2]:
# ============================================================
# AUTO-UPDATE: Layer 0–6 are executed before Layer 7
# ============================================================
import subprocess, sys

LAYER_NOTEBOOKS = [
    "layer0_external_drivers.ipynb",
    "layer1_planetary_body.ipynb",
    "layer2_surface_zone.ipynb",
    "layer3_atmosphere_zone.ipynb",
    "layer4_Ionosphere.ipynb",
    "layer5_Global_Electric_Circuit.ipynb",
    "layer6_Resonance_Field.ipynb",
]

for nb in LAYER_NOTEBOOKS:
    print(f"▶ Updating {nb} ...")
    result = subprocess.run(
        ["jupyter", "nbconvert", "--to", "notebook",
         "--execute", nb, "--inplace",
         "--ExecutePreprocessor.timeout=300"],
        capture_output=True, text=True
    )
    if result.returncode == 0:
        print(f"  ✅ {nb} done")
    else:
        print(f"  ⚠️  {nb} failed – Layer 7 uses last known state")
        print(f"     {result.stderr[-200:]}")

print("\n🌐 All layers updated – starting Layer 7 Engine...\n")

▶ Updating layer0_external_drivers.ipynb ...


  ✅ layer0_external_drivers.ipynb done
▶ Updating layer1_planetary_body.ipynb ...


  ✅ layer1_planetary_body.ipynb done
▶ Updating layer2_surface_zone.ipynb ...


  ✅ layer2_surface_zone.ipynb done
▶ Updating layer3_atmosphere_zone.ipynb ...


  ✅ layer3_atmosphere_zone.ipynb done
▶ Updating layer4_Ionosphere.ipynb ...


  ✅ layer4_Ionosphere.ipynb done
▶ Updating layer5_Global_Electric_Circuit.ipynb ...


  ✅ layer5_Global_Electric_Circuit.ipynb done
▶ Updating layer6_Resonance_Field.ipynb ...


  ✅ layer6_Resonance_Field.ipynb done

🌐 All layers updated – starting Layer 7 Engine...



In [3]:
# ============================================================
# READ ALL LAYER STATES
# ============================================================
layers = {}
for n in range(7):
    fp = f'layer{n}_test_state.json'
    if os.path.exists(fp):
        try:
            with open(fp, encoding='utf-8') as f:
                layers[n] = json.load(f)
        except Exception as e:
            print(f'  L{n}: error {e}')
            layers[n] = None
    else:
        print(f'  L{n}: file missing')
        layers[n] = None

print('LAYER STATE OVERVIEW')
print('=' * 78)
print(f'  {"Layer":<6} {"Score":>7} {"Level":>10} {"Conf":>6} {"Dominant":<32}')
print('-' * 78)
for n, L in layers.items():
    if L:
        score = L.get('score', 0) or 0
        level = L.get('level', 'unknown')
        conf  = L.get('confidence', 0) or 0
        dom   = L.get('dominant_component') or L.get('dominant_driver') or '–'
        print(f'  L{n:<5} {score:>7.3f} {level:>10} {conf:>6.0%} {dom[:30]:<32}')
    else:
        print(f'  L{n:<5} {"–":>7} {"missing":>10} {"–":>6} {"–":<32}')
print('=' * 78)

LAYER STATE OVERVIEW
  Layer    Score      Level   Conf Dominant                        
------------------------------------------------------------------------------
  L0       0.147       calm    75% none                            
  L1       0.292       calm   100% –                               
  L2       0.359   moderate    80% ENSO Phase (Nino3.4)            
  L3       0.140      quiet   100% Atm. Instability (LI)           
  L4       0.305   moderate   100% X-Ray Absorption (D-Layer)      
  L5       0.273      quiet   100% Iono. Conductivity (L4)         
  L6       0.251      quiet   100% Excitation Strength (Amplitude  


---
## 2.Normalization: uniform layer state per layer

In [4]:
# ============================================================
# EACH LAYER IS NORMALIZED TO A UNIFORM STRUCTURE
# {score, level, confidence, dominant_component, flags, key_metrics}
# ============================================================
LAYER_NAMES = {
    0: 'L0_external_drivers',
    1: 'L1_planetary_body',
    2: 'L2_surface_zone',
    3: 'L3_atmosphere',
    4: 'L4_ionosphere',
    5: 'L5_global_electric_circuit',
    6: 'L6_resonance_field',
}

def extract_key_metrics(n, L):
    """Extracts the most important metrics per layer"""
    if not L: return {}
    rv = L.get('raw_values', {})
    if n == 0:
        return {'F10.7': rv.get('F10.7_sfu', {}).get('value') if isinstance(rv.get('F10.7_sfu'), dict) else rv.get('F10.7_sfu'),
                'Kp':    rv.get('Kp_index', {}).get('value') if isinstance(rv.get('Kp_index'), dict) else rv.get('Kp_index'),
                'IMF_Bz': rv.get('IMF_Bz_nT', {}).get('value') if isinstance(rv.get('IMF_Bz_nT'), dict) else rv.get('IMF_Bz_nT')}
    if n == 1:
        return {'seismic_events': rv.get('seismic_events_7d'),
                'max_mag':        rv.get('seismic_max_mag'),
                'LOD_anomaly':    rv.get('LOD_anomaly_ms')}
    if n == 2:
        enso = rv.get('ENSO', {})
        return {'SST_anomaly':  rv.get('SST_anomaly_degC'),
                'ENSO_phase':   enso.get('phase_observed'),
                'conv_pot':     rv.get('conv_potential_mean')}
    if n == 3:
        return {'CAPE':         rv.get('CAPE_mean_Jkg'),
                'thunder_pts':  rv.get('thunder_points_WMO'),
                'storms_EONET': rv.get('storm_events_EONET')}
    if n == 4:
        rs = L.get('resonance_system', {})
        return {'Kp':         rv.get('Kp_current'),
                'cavity_h':   rs.get('cavity_height_km'),
                'xray_class': rv.get('xray_class')}
    if n == 5:
        gec = L.get('gec_state', {})
        return {'V_iono':         gec.get('V_ionosphere_kV'),
                'delta_V_pct':    gec.get('delta_V_pct'),
                'generator':      gec.get('generator_strength')}
    if n == 6:
        em = L.get('expected_modulated', {}).get('SR_1', {})
        da = L.get('delta_analysis', {})
        return {'SR1_Hz':           em.get('freq_Hz'),
                'SR1_amp_pT':       em.get('amplitude_pT'),
                'non_geom_ratio':   da.get('ratio_non_geom_to_geom')}
    return {}

normalized = {}
for n in range(7):
    L = layers.get(n)
    if not L:
        normalized[LAYER_NAMES[n]] = {
            'score': None, 'level': 'unknown', 'confidence': 0.0,
            'dominant_component': None, 'flags': {}, 'key_metrics': {},
            'available': False,
        }
        continue
    score = L.get('score')
    normalized[LAYER_NAMES[n]] = {
        'score':              score,
        'level':              classify_level(score) if score is not None else L.get('level', 'unknown'),
        'confidence':         L.get('confidence', 0.0) or 0.0,
        'dominant_component': L.get('dominant_component') or L.get('dominant_driver'),
        'flags':              L.get('flags', {}),
        'key_metrics':        extract_key_metrics(n, L),
        'available':          True,
    }

available_layers = [k for k, v in normalized.items() if v['available']]
print(f'Available layers: {len(available_layers)}/7')
for name, st in normalized.items():
    if st['available']:
        print(f'  {name:<32} score={st["score"]:.3f}  conf={st["confidence"]:.0%}  level={st["level"]}')

Available layers: 7/7
  L0_external_drivers              score=0.147  conf=75%  level=quiet
  L1_planetary_body                score=0.292  conf=100%  level=quiet
  L2_surface_zone                  score=0.359  conf=80%  level=moderate
  L3_atmosphere                    score=0.140  conf=100%  level=quiet
  L4_ionosphere                    score=0.305  conf=100%  level=moderate
  L5_global_electric_circuit       score=0.273  conf=100%  level=quiet
  L6_resonance_field               score=0.251  conf=100%  level=quiet


---
## 3.Trend analysis from history

In [5]:
# ============================================================
# READ HISTORY AND CALCULATE DELTAS
# ============================================================
history = []
if os.path.exists(HISTORY_FILE):
    with open(HISTORY_FILE, encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if line:
                try:
                    history.append(json.loads(line))
                except: continue

print(f'History: {len(history)} snapshots loaded')

def find_snapshot_at_age(history, hours_ago, tolerance_hours=2):
    """Finds snapshot that is ~hours_ago old"""
    target = datetime.datetime.utcnow() - datetime.timedelta(hours=hours_ago)
    best = None; best_diff = float('inf')
    for snap in history:
        try:
            t = datetime.datetime.fromisoformat(snap['timestamp'].replace('Z', ''))
            diff = abs((t - target).total_seconds() / 3600)
            if diff < best_diff and diff < tolerance_hours:
                best = snap; best_diff = diff
        except: continue
    return best

snap_1h  = find_snapshot_at_age(history, 1,  tolerance_hours=1.5)
snap_6h  = find_snapshot_at_age(history, 6,  tolerance_hours=3)
snap_24h = find_snapshot_at_age(history, 24, tolerance_hours=6)

trends = {}
for name, st in normalized.items():
    if not st['available'] or st['score'] is None:
        trends[name] = {'delta_1h': None, 'delta_6h': None, 'delta_24h': None,
                        'trend': 'unknown', 'volatility': None}
        continue
    cur = st['score']
    d_1h  = round(cur - snap_1h['layers'][name]['score'], 4)  if snap_1h  and snap_1h.get('layers',{}).get(name,{}).get('score') is not None else None
    d_6h  = round(cur - snap_6h['layers'][name]['score'], 4)  if snap_6h  and snap_6h.get('layers',{}).get(name,{}).get('score') is not None else None
    d_24h = round(cur - snap_24h['layers'][name]['score'], 4) if snap_24h and snap_24h.get('layers',{}).get(name,{}).get('score') is not None else None

    # Trend category from 6h delta (or 24h if 6h missing)
    delta_for_trend = d_6h if d_6h is not None else d_24h
    if delta_for_trend is None:
        trend = 'unknown'
    elif delta_for_trend > 0.05:  trend = 'rising'
    elif delta_for_trend < -0.05: trend = 'falling'
    else:                         trend = 'stable'

    # Volatility: standard deviation of last 10 snapshots
    recent_scores = [s['layers'].get(name,{}).get('score') for s in history[-10:]
                     if s.get('layers',{}).get(name,{}).get('score') is not None]
    volatility = round(float(np.std(recent_scores)), 4) if len(recent_scores) >= 3 else None

    trends[name] = {
        'delta_1h':   d_1h,
        'delta_6h':   d_6h,
        'delta_24h':  d_24h,
        'trend':      trend,
        'volatility': volatility,
    }

print('\nTrend overview:')
for name, t in trends.items():
    if normalized[name]['available']:
        d1  = f'{t["delta_1h"]:+.3f}'  if t['delta_1h']  is not None else '   –   '
        d6  = f'{t["delta_6h"]:+.3f}'  if t['delta_6h']  is not None else '   –   '
        d24 = f'{t["delta_24h"]:+.3f}' if t['delta_24h'] is not None else '   –   '
        v   = f'{t["volatility"]:.3f}' if t['volatility'] is not None else '   –   '
        print(f'  {name:<32}  Δ1h={d1}  Δ6h={d6}  Δ24h={d24}  vol={v}  [{t["trend"]}]')

History: 8 snapshots loaded

Trend overview:
  L0_external_drivers               Δ1h=   –     Δ6h=   –     Δ24h=-0.001  vol=0.026  [stable]
  L1_planetary_body                 Δ1h=   –     Δ6h=   –     Δ24h=+0.002  vol=0.024  [stable]
  L2_surface_zone                   Δ1h=   –     Δ6h=   –     Δ24h=-0.066  vol=0.082  [falling]
  L3_atmosphere                     Δ1h=   –     Δ6h=   –     Δ24h=-0.015  vol=0.022  [stable]
  L4_ionosphere                     Δ1h=   –     Δ6h=   –     Δ24h=+0.006  vol=0.029  [stable]
  L5_global_electric_circuit        Δ1h=   –     Δ6h=   –     Δ24h=+0.030  vol=0.040  [stable]
  L6_resonance_field                Δ1h=   –     Δ6h=   –     Δ24h=-0.021  vol=0.013  [stable]


---
## 4.Coupling calculation between layers

In [6]:
# ============================================================
# COUPLING MATRIX
# Known physical couplings with strength derived from data
# ============================================================

def safe(v, default=0.0):
    return float(v) if v is not None else default

# Helper lookups
L0 = layers.get(0); L1 = layers.get(1); L2 = layers.get(2)
L3 = layers.get(3); L4 = layers.get(4); L5 = layers.get(5); L6 = layers.get(6)

# Defined couplings with calculation logic
couplings = []

# L0 → L4 (Space Weather → Ionosphere)
if L0 and L4:
    s0 = safe(L0.get('score'))
    s4 = safe(L4.get('score'))
    # Strength: correlated fraction
    strength = round(min(s0, s4) * 0.9 + abs(s0 - s4) * 0.1, 3)
    couplings.append({
        'from': 'L0_external_drivers', 'to': 'L4_ionosphere',
        'strength': strength, 'lag_hours': 0,
        'confidence': round(min(L0.get('confidence',0), L4.get('confidence',0)), 2),
        'mechanism': 'Solar wind, F10.7, X-Ray ionize ionosphere'
    })

# L3 → L5 (Thunderstorms → GEC Generator)
if L3 and L5:
    thunder_score = safe(L3.get('components',{}).get('Thunderstorm Activity',{}).get('score'))
    generator     = safe(L5.get('gec_state',{}).get('generator_strength', 1.0)) - 1.0
    strength = round(min(1.0, abs(generator) * 2 + thunder_score * 0.5), 3)
    couplings.append({
        'from': 'L3_atmosphere', 'to': 'L5_global_electric_circuit',
        'strength': strength, 'lag_hours': 0,
        'confidence': round(min(L3.get('confidence',0), L5.get('confidence',0)), 2),
        'mechanism': 'Thunderstorms charge ionosphere (CAPE → V_iono)'
    })

# L3 → L6 (Lightning activity → Schumann amplitude)
if L3 and L6:
    thunder_score = safe(L3.get('components',{}).get('Thunderstorm Activity',{}).get('score'))
    amp_factor    = safe(L6.get('modulators',{}).get('amplitude_factor',{}).get('thunder_l3', 1.0)) - 1.0
    strength = round(min(1.0, thunder_score * 0.7 + abs(amp_factor) * 3), 3)
    couplings.append({
        'from': 'L3_atmosphere', 'to': 'L6_resonance_field',
        'strength': strength, 'lag_hours': 0,
        'confidence': round(min(L3.get('confidence',0), L6.get('confidence',0)), 2),
        'mechanism': 'Lightning excites Schumann resonance'
    })

# L4 → L6 (Cavity → Resonance conditions)
if L4 and L6:
    cav_dev = abs(safe(L4.get('resonance_system',{}).get('cavity_delta_km'))) / 20
    ioniz   = safe(L4.get('components',{}).get('Ionization Level (F10.7)',{}).get('score'))
    strength = round(min(1.0, cav_dev * 0.5 + ioniz * 0.5), 3)
    couplings.append({
        'from': 'L4_ionosphere', 'to': 'L6_resonance_field',
        'strength': strength, 'lag_hours': 0,
        'confidence': round(min(L4.get('confidence',0), L6.get('confidence',0)), 2),
        'mechanism': 'Cavity height + conductivity modulate frequency and Q'
    })

# L5 → L6 (GEC potential → Resonance energy)
if L5 and L6:
    delta_v_pct = abs(safe(L5.get('gec_state',{}).get('delta_V_pct'))) / 30
    s5 = safe(L5.get('score'))
    strength = round(min(1.0, delta_v_pct * 0.6 + s5 * 0.4), 3)
    couplings.append({
        'from': 'L5_global_electric_circuit', 'to': 'L6_resonance_field',
        'strength': strength, 'lag_hours': 0,
        'confidence': round(min(L5.get('confidence',0), L6.get('confidence',0)), 2),
        'mechanism': 'GEC is the electrical architecture of resonance'
    })

# L0 → L5 (Geomagnetic modulation of GEC)
if L0 and L5:
    kp_score = safe(L0.get('components',{}).get('Geomag. Activity (Kp)'))
    s5 = safe(L5.get('score'))
    strength = round(min(1.0, kp_score * 0.6 + s5 * 0.2), 3)
    couplings.append({
        'from': 'L0_external_drivers', 'to': 'L5_global_electric_circuit',
        'strength': strength, 'lag_hours': 0,
        'confidence': round(min(L0.get('confidence',0), L5.get('confidence',0)), 2),
        'mechanism': 'Kp modulates ionospheric conductivity (GEC resistance)'
    })

# L2 → L3 (Convection → Thunderstorms)
if L2 and L3:
    conv = safe(L2.get('components',{}).get('Convection Potential',{}).get('score'))
    s3 = safe(L3.get('score'))
    strength = round(min(1.0, conv * 0.7 + s3 * 0.3), 3)
    couplings.append({
        'from': 'L2_surface_zone', 'to': 'L3_atmosphere',
        'strength': strength, 'lag_hours': 1,
        'confidence': round(min(L2.get('confidence',0), L3.get('confidence',0)), 2),
        'mechanism': 'Surface convection → atmospheric thunderstorm formation'
    })

print('COUPLING MATRIX')
print('=' * 78)
for c in couplings:
    arrow = ('▰' * int(c['strength']*10) + '▱' * (10 - int(c['strength']*10)))
    print(f'  {c["from"]:<28} → {c["to"]:<26}  {arrow}  {c["strength"]:.3f}')
print('=' * 78)

COUPLING MATRIX
  L0_external_drivers          → L4_ionosphere               ▰▱▱▱▱▱▱▱▱▱  0.148
  L3_atmosphere                → L5_global_electric_circuit  ▰▱▱▱▱▱▱▱▱▱  0.195
  L3_atmosphere                → L6_resonance_field          ▰▰▰▱▱▱▱▱▱▱  0.347
  L4_ionosphere                → L6_resonance_field          ▰▰▰▰▱▱▱▱▱▱  0.408
  L5_global_electric_circuit   → L6_resonance_field          ▰▰▱▱▱▱▱▱▱▱  0.225
  L0_external_drivers          → L5_global_electric_circuit  ▱▱▱▱▱▱▱▱▱▱  0.055
  L2_surface_zone              → L3_atmosphere               ▰▰▰▱▱▱▱▱▱▱  0.350


---
## 5.Dominant Layer & System State Classification

In [7]:
# ============================================================
# IDENTIFY DOMINANT LAYERS
# ============================================================
scored = [(name, st['score']) for name, st in normalized.items()
          if st['available'] and st['score'] is not None]
scored_sorted = sorted(scored, key=lambda x: -x[1])

dominant_layer  = scored_sorted[0][0]   if len(scored_sorted) >= 1 else None
secondary_layer = scored_sorted[1][0]   if len(scored_sorted) >= 2 else None
weak_layers     = [n for n, s in scored_sorted if s < 0.3]
active_layers   = [n for n, s in scored_sorted if s >= 0.5]

print(f'Dominant layer:    {dominant_layer}  ({normalized[dominant_layer]["score"]:.3f})')
print(f'Secondary layer:   {secondary_layer} ({normalized[secondary_layer]["score"]:.3f})' if secondary_layer else '')
print(f'Active layers:     {active_layers}')
print(f'Quiet layers:      {weak_layers}')

# ============================================================
# SYSTEM STATE CLASSIFICATION
# ============================================================
def classify_system_state(normalized, couplings, dominant, active):
    """Classifies the current Earth field state"""
    avg_score = np.mean([st['score'] for st in normalized.values() if st['score'] is not None])
    avg_conf  = np.mean([st['confidence'] for st in normalized.values() if st['available']])

    L0 = normalized.get('L0_external_drivers', {})
    L3 = normalized.get('L3_atmosphere', {})
    L4 = normalized.get('L4_ionosphere', {})
    L6 = normalized.get('L6_resonance_field', {})

    # Low confidence
    if avg_conf < 0.5:
        return 'low_confidence_state', avg_score, 0.4

    # Anomalous resonance has highest priority
    if L6.get('flags', {}).get('non_geometric_dominant'):
        if L6.get('flags', {}).get('frequency_anomaly') or L6.get('flags', {}).get('amplitude_elevated'):
            return 'anomalous_resonance_state', L6['score'] or 0, 0.75

    # Geomagnetic disturbance
    if L0.get('flags', {}).get('geomagnetic_storm') or L4.get('flags', {}).get('geomagnetic_storm'):
        return 'geomagnetic_disturbance_state', max(L0['score'] or 0, L4['score'] or 0), 0.85

    # Atmospherically driven resonance: L3 + L6 dominant
    if dominant in ['L3_atmosphere', 'L6_resonance_field'] and 'L3_atmosphere' in active and 'L6_resonance_field' in active:
        return 'atmospheric_driven_resonance_state', (L3['score'] + L6['score']) / 2, 0.78

    # Space weather driven: L0 + L4 dominant
    if dominant in ['L0_external_drivers', 'L4_ionosphere'] and ('L0_external_drivers' in active or 'L4_ionosphere' in active):
        s = (safe(L0.get('score')) + safe(L4.get('score'))) / 2
        return 'space_weather_driven_ionospheric_state', s, 0.80

    # Mixed
    if len(active) >= 3:
        strong_couplings = [c for c in couplings if c['strength'] > 0.5]
        if len(strong_couplings) >= 2:
            return 'mixed_coupled_state', avg_score, 0.70

    # Cavity shift as main driver
    if L4.get('flags', {}).get('cavity_elevated'):
        return 'cavity_condition_shift_state', L4['score'] or 0, 0.65

    # Seasonal: ENSO active
    L2 = normalized.get('L2_surface_zone', {})
    if L2.get('flags', {}).get('el_nino_developing') or L2.get('flags', {}).get('la_nina_active'):
        return 'seasonal_transition_state', L2['score'] or 0, 0.65

    # Default
    return 'normal_background_state', avg_score, 0.85

system_state, state_score, state_confidence = classify_system_state(
    normalized, couplings, dominant_layer, active_layers
)

print(f'\nSystem state:      {system_state}')
print(f'State score:       {state_score:.3f}')
print(f'State confidence:  {state_confidence:.0%}')

Dominant layer:    L2_surface_zone  (0.359)
Secondary layer:   L4_ionosphere (0.305)
Active layers:     []
Quiet layers:      ['L1_planetary_body', 'L5_global_electric_circuit', 'L6_resonance_field', 'L0_external_drivers', 'L3_atmosphere']

System state:      seasonal_transition_state
State score:       0.359
State confidence:  65%


---
## 6.Event tag generation

In [8]:
# ============================================================
# AUTOMATIC EVENT TAGS
# These are the most important Layer 8 research material
# ============================================================
def gen_event_tags(layers, normalized, system_state):
    tags = []
    L0 = layers.get(0); L1 = layers.get(1); L2 = layers.get(2)
    L3 = layers.get(3); L4 = layers.get(4); L5 = layers.get(5); L6 = layers.get(6)

    # L0
    if L0 and L0.get('flags', {}).get('geomagnetic_storm'):
        tags.append('geomagnetic_storm')
    if L0 and L0.get('flags', {}).get('solar_flux_high'):
        tags.append('high_solar_flux')
    if L0 and L0.get('dominant_driver') and L0['dominant_driver'] != 'none':
        tags.append('space_weather_influence')

    # L1
    if L1 and L1.get('flags', {}).get('elevated_seismicity'):
        tags.append('elevated_seismicity')
    if L1 and L1.get('flags', {}).get('strong_earthquake'):
        tags.append('strong_earthquake')

    # L2
    if L2 and L2.get('flags', {}).get('el_nino_developing'):
        tags.append('el_nino_developing')
    if L2 and L2.get('flags', {}).get('la_nina_active'):
        tags.append('la_nina_active')
    if L2 and L2.get('flags', {}).get('thunderstorm_trigger'):
        tags.append('surface_convection_trigger')
    if L2 and L2.get('flags', {}).get('sst_anomaly_high'):
        tags.append('sst_anomaly_high')

    # L3
    if L3 and L3.get('flags', {}).get('active_thunderstorms'):
        tags.append('high_thunderstorm_activity')
    if L3 and L3.get('flags', {}).get('high_cape'):
        tags.append('high_cape')
    if L3 and L3.get('flags', {}).get('atmospheric_instability'):
        tags.append('atmospheric_instability')

    # L4
    if L4 and L4.get('flags', {}).get('ionospheric_disturbed'):
        tags.append('ionospheric_disturbance')
    if L4 and L4.get('flags', {}).get('radio_blackout'):
        tags.append('radio_blackout')
    if L4 and L4.get('flags', {}).get('cavity_elevated'):
        tags.append('cavity_condition_shift')
    if L4 and L4.get('flags', {}).get('bz_southward_coupled'):
        tags.append('imf_bz_coupling')

    # L5
    if L5 and L5.get('flags', {}).get('gec_elevated'):
        tags.append('gec_elevated')
    if L5 and L5.get('flags', {}).get('gec_suppressed'):
        tags.append('gec_suppressed')

    # L6
    if L6 and L6.get('flags', {}).get('amplitude_elevated'):
        tags.append('elevated_resonance')
    if L6 and L6.get('flags', {}).get('frequency_anomaly'):
        tags.append('schumann_freq_anomaly')
    if L6 and L6.get('flags', {}).get('non_geometric_dominant'):
        tags.append('non_geometric_dominance')
    if L6 and L6.get('flags', {}).get('q_factor_degraded'):
        tags.append('q_factor_degraded')

    # Global tags
    avg_score = np.mean([st['score'] for st in normalized.values() if st['score'] is not None])
    if avg_score < 0.3:
        tags.append('low_global_activity')
    elif avg_score > 0.6:
        tags.append('high_global_activity')

    # System state as tag
    tags.append(f'state_{system_state}')
    return sorted(set(tags))

event_tags = gen_event_tags(layers, normalized, system_state)
print(f'Event tags ({len(event_tags)}):')
for t in event_tags:
    print(f'  • {t}')

Event tags (5):
  • el_nino_developing
  • elevated_seismicity
  • low_global_activity
  • non_geometric_dominance
  • state_seasonal_transition_state


---
## 7.visualizations

In [9]:
# ============================================================
# LAYER OVERVIEW: Score, Trend, Confidence
# ============================================================
names = list(normalized.keys())
scores = [st['score'] if st['score'] is not None else 0 for st in normalized.values()]

trend_arrows = []
for n in names:
    t = trends.get(n, {}).get('trend', 'unknown')
    trend_arrows.append({'rising':'↑','falling':'↓','stable':'→','unknown':'?'}[t])

score_colors = ['#2ecc71' if s < 0.3 else '#f39c12' if s < 0.6 else '#e74c3c' if s < 0.8 else '#c0392b'
                for s in scores]

fig = go.Figure()
fig.add_trace(go.Bar(
    x=scores, y=names, orientation='h',
    marker_color=score_colors, opacity=0.85,
    text=[f'<b>{s:.3f}</b> {a}' for s, a in zip(scores, trend_arrows)],
    textposition='outside',
    textfont=dict(color='white', size=11)
))

fig.add_vline(x=0.3, line_dash='dot', line_color='#888780', annotation_text='quiet')
fig.add_vline(x=0.6, line_dash='dot', line_color='#f39c12', annotation_text='moderate')
fig.add_vline(x=0.8, line_dash='dot', line_color='#e74c3c', annotation_text='active')

fig.update_layout(
    title=dict(text=f'Layer Overview | System State: <b>{system_state}</b> | Conf {state_confidence:.0%}',
               font=dict(size=14)),
    xaxis=dict(title='Score [0–1]', range=[0, 1.15], gridcolor='#222244'),
    height=400, plot_bgcolor='#1a1a2e', paper_bgcolor='rgba(0,0,0,0)',
    margin=dict(l=200, r=80, t=55, b=40), showlegend=False
)
fig.show()

In [10]:
# ============================================================
# COUPLING MATRIX – Heatmap
# ============================================================
all_layers = list(LAYER_NAMES.values())
matrix = np.zeros((7, 7))
for c in couplings:
    i = all_layers.index(c['from'])
    j = all_layers.index(c['to'])
    matrix[i, j] = c['strength']

labels_short = [n.replace('_', '<br>', 1) for n in all_layers]

fig = go.Figure(go.Heatmap(
    z=matrix, x=labels_short, y=labels_short,
    colorscale=[[0, '#1a1a2e'], [0.3, '#534AB7'], [0.6, '#F2A623'], [1.0, '#e74c3c']],
    zmin=0, zmax=1,
    text=[[f'{v:.2f}' if v > 0 else '' for v in row] for row in matrix],
    texttemplate='%{text}',
    textfont=dict(size=10, color='white'),
    colorbar=dict(title='Strength')
))

fig.update_layout(
    title=dict(text='Coupling Matrix: Who influences whom? (Row → Column)', font=dict(size=14)),
    height=480, plot_bgcolor='#1a1a2e', paper_bgcolor='rgba(0,0,0,0)',
    margin=dict(l=120, r=80, t=55, b=120),
    xaxis=dict(side='bottom', tickangle=30),
    yaxis=dict(autorange='reversed')
)
fig.show()

In [11]:
# ============================================================
# LAYER 7 DASHBOARD – 4 Charts
# ============================================================

snap_current = {
    'timestamp':    RUN_TIMESTAMP,
    'system_state': system_state,
    'layers': {n: {'score': normalized[n]['score']} for n in LAYER_NAMES.values()}
}
all_snaps = history + [snap_current]
n_snaps   = len(all_snaps)

layer_list   = list(LAYER_NAMES.values())
layer_labels = [
    'L0 External Drivers',
    'L1 Planetary Body',
    'L2 Surface Zone',
    'L3 Atmosphere',
    'L4 Ionosphere',
    'L5 GEC',
    'L6 Resonance Field',
]

STATE_COLORS = {
    'normal_background_state':               '#2ecc71',
    'atmospheric_driven_resonance_state':    '#E85D24',
    'space_weather_driven_ionospheric_state':'#534AB7',
    'geomagnetic_disturbance_state':         '#e74c3c',
    'mixed_coupled_state':                  '#F2A623',
    'seasonal_transition_state':            '#639922',
    'cavity_condition_shift_state':         '#378ADD',
    'anomalous_resonance_state':            '#c0392b',
    'low_confidence_state':                 '#888780',
}
STATE_SHORT = {
    'normal_background_state':               'normal',
    'atmospheric_driven_resonance_state':    'atmospheric',
    'space_weather_driven_ionospheric_state':'space_weather',
    'geomagnetic_disturbance_state':         'geomagnetic',
    'mixed_coupled_state':                  'mixed',
    'seasonal_transition_state':            'seasonal',
    'cavity_condition_shift_state':         'cavity_shift',
    'anomalous_resonance_state':            'anomalous',
    'low_confidence_state':                 'low_conf',
}

times_str = [s['timestamp'][:16].replace('T', ' ') for s in all_snaps]

from plotly.subplots import make_subplots
fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=[
        'Layer Scores Heatmap',
        'System State Timeline',
        'Layer Delta / Change  (Δ since previous snapshot)',
        'Current Dominance Ranking',
    ],
    vertical_spacing=0.20,
    horizontal_spacing=0.12,
    row_heights=[0.52, 0.48],
)

# ── Chart 1: Heatmap ─────────────────────────────────────────
z_matrix = []
for ln in layer_list:
    row = []
    for snap in all_snaps:
        v = snap.get('layers', {}).get(ln, {}).get('score')
        row.append(float(v) if v is not None else 0.0)
    z_matrix.append(row)

txt_matrix = [[f'{v:.2f}' for v in row] for row in z_matrix]

fig.add_trace(go.Heatmap(
    z=z_matrix, x=times_str, y=layer_labels,
    colorscale=[[0,'#1a1a2e'],[0.25,'#378ADD'],[0.5,'#F2A623'],[0.75,'#e74c3c'],[1.0,'#c0392b']],
    zmin=0, zmax=1,
    text=txt_matrix, texttemplate='%{text}', textfont=dict(size=10, color='white'),
    showscale=True,
    colorbar=dict(len=0.42, y=0.76, x=0.46, thickness=12,
                  tickvals=[0, 0.25, 0.5, 0.75, 1.0],
                  ticktext=['0', '.25 quiet', '.50 moderate', '.75 active', '1'],
                  tickfont=dict(color='white', size=9))
), row=1, col=1)

# ── Chart 2: State Timeline ───────────────────────────────────
state_clrs  = [STATE_COLORS.get(s.get('system_state',''), '#888780') for s in all_snaps]
state_txts  = [STATE_SHORT.get(s.get('system_state',''), '?') for s in all_snaps]

fig.add_trace(go.Bar(
    x=times_str, y=[1] * n_snaps,
    marker_color=state_clrs, opacity=0.88,
    text=state_txts,
    textposition='inside',
    textfont=dict(size=10, color='white'),
    showlegend=False
), row=1, col=2)
fig.update_yaxes(visible=False, row=1, col=2)
fig.update_xaxes(tickangle=30, row=1, col=2)

# ── Chart 3: Delta ───────────────────────────────────────────
delta_available = any(trends.get(ln, {}).get('delta_6h') is not None for ln in layer_list)

if delta_available:
    d_vals  = [trends.get(ln, {}).get('delta_6h') or 0.0 for ln in layer_list]
    d_clrs  = ['#e74c3c' if v > 0.05 else '#378ADD' if v < -0.05 else '#888780' for v in d_vals]
    d_texts = [f'{v:+.3f}' for v in d_vals]
else:
    d_vals  = [normalized[ln].get('score') or 0 for ln in layer_list]
    d_clrs  = ['#444466'] * 7
    d_texts = [f'{v:.3f} (cur)' for v in d_vals]

fig.add_trace(go.Bar(
    x=d_vals, y=layer_labels,
    orientation='h',
    marker_color=d_clrs, opacity=0.87,
    text=d_texts,
    textposition='outside', textfont=dict(color='white', size=10),
    showlegend=False
), row=2, col=1)
fig.add_vline(x=0, line_color='#555588', line_width=1, row=2, col=1)
fig.add_vline(x=0.05,  line_dash='dot', line_color='#e74c3c', line_width=1, row=2, col=1)
fig.add_vline(x=-0.05, line_dash='dot', line_color='#378ADD', line_width=1, row=2, col=1)
fig.update_xaxes(
    range=[-0.35, 0.42] if delta_available else [0, 1.1],
    title_text='Δ Score (6h)' if delta_available else 'Current Score (Δ from 2nd snapshot)',
    gridcolor='#222244', row=2, col=1
)
if not delta_available:
    fig.add_annotation(
        x=0.55, y=6.6,
        text='⚠ Delta available after 2 snapshots',
        showarrow=False, xref='x3', yref='y3',
        font=dict(color='#F2A623', size=10),
        bgcolor='rgba(30,30,50,0.8)'
    )

# ── Chart 4: Dominance Ranking ───────────────────────────────
coup_pairs  = [f"{c['from'].split('_')[0]} → {c['to'].split('_')[0]}" for c in couplings]
coup_vals   = [c['strength'] for c in couplings]
coup_clrs   = ['#e74c3c' if v > 0.5 else '#F2A623' if v > 0.3 else '#534AB7' for v in coup_vals]
sorted_coup = sorted(zip(coup_pairs, coup_vals, coup_clrs), key=lambda x: x[1])

fig.add_trace(go.Bar(
    x=[v for _, v, _ in sorted_coup],
    y=[l for l, _, _ in sorted_coup],
    orientation='h',
    marker_color=[c for _, _, c in sorted_coup], opacity=0.88,
    text=[f'{v:.2f}' for _, v, _ in sorted_coup],
    textposition='outside', textfont=dict(color='white', size=10),
    showlegend=False
), row=2, col=2)
for t, col, label in [(0.25,'#555588','quiet'),(0.50,'#888780','moderate'),(0.75,'#e74c3c','strong')]:
    fig.add_vline(x=t, line_dash='dot', line_color=col, line_width=1,
                  annotation_text=label,
                  annotation_font=dict(color=col, size=8),
                  annotation_position='top',
                  row=2, col=2)
fig.update_xaxes(range=[0, 1.12], title_text='Coupling strength', gridcolor='#222244', row=2, col=2)

# ── Global State Header ───────────────────────────────────────
fig.add_annotation(
    x=0.5, y=1.06, xref='paper', yref='paper',
    text=(
        f'<b>Earth Field State:</b> {STATE_SHORT.get(system_state, system_state)}  '
        f'  <b>Dominant Layer:</b> {dominant_layer.split("_",1)[-1].replace("_"," ").title()}  '
        f'  <b>Score:</b> {state_score:.3f}'
        f'  <b>Confidence:</b> {state_confidence:.0%}'
        f'  <b>Snapshots:</b> {n_snaps}'
    ),
    showarrow=False, font=dict(size=12, color='white'),
    bgcolor='rgba(30,30,60,0.85)',
    bordercolor='#534AB7', borderwidth=1,
    xanchor='center'
)

# ── Layout ────────────────────────────────────────────────────
fig.update_layout(
    height=680,
    plot_bgcolor='#1a1a2e', paper_bgcolor='rgba(0,0,0,0)',
    margin=dict(l=150, r=80, t=100, b=70),
    showlegend=False,
    font=dict(color='white', size=11),
)
for r, c in [(1,1),(1,2),(2,1),(2,2)]:
    fig.update_yaxes(gridcolor='#222244', tickfont=dict(color='white', size=10), row=r, col=c)
    fig.update_xaxes(gridcolor='#222244', tickfont=dict(color='white', size=9), row=r, col=c)

fig.show()

---
## 8.Export & Archive

In [12]:
# ============================================================
# CURRENT STATE – layer7_test_state.json
# ============================================================

avg_score      = round(float(np.mean([st['score'] for st in normalized.values() if st['score'] is not None])), 4)
avg_confidence = round(float(np.mean([st['confidence'] for st in normalized.values() if st['available']])), 4)

earth_field_state = {
    'timestamp':       RUN_TIMESTAMP,
    'engine_version':  '1.0',
    'layer':           7,
    'name':            'Earth Field State Engine',

    # Global state
    'system_state':    system_state,
    'state_score':           round(float(state_score), 4),
    'state_score_method':    'dominant_layer_score',
    'state_confidence':      round(float(state_confidence), 2),
    'avg_score':             avg_score,
    'avg_score_method':      'mean_of_available_layers',
    'avg_confidence':        avg_confidence,

    # State confidence reasoning
    'state_confidence_reason': (
        dominant_layer + ' is active/dominant, but downstream confirmation'
        ' in L3_atmosphere (score=' + f'{normalized["L3_atmosphere"]["score"]:.3f}' + ')'
        ' and L6_resonance_field (score=' + f'{normalized["L6_resonance_field"]["score"]:.3f}' + ')'
        ' is ' + ('strong' if (normalized['L6_resonance_field']['score'] or 0) > 0.4 else 'weak') + '.'
    ),

    # Dominance analysis
    'dominance': {
        'dominant_layer':   dominant_layer,
        'secondary_layer':  secondary_layer,
        'active_layers':    active_layers,
        'weak_layers':      weak_layers,
    },

    # Per layer
    'layers': {
        name: {
            **st,
            **trends.get(name, {})
        }
        for name, st in normalized.items()
    },

    # Diagnostic features – for Layer 8 research
    'diagnostic_features': (lambda: {
        'surface_atmosphere_gap':    round(
            (normalized['L2_surface_zone']['score'] or 0) -
            (normalized['L3_atmosphere']['score'] or 0), 4),
        'surface_to_resonance_gap':  round(
            (normalized['L2_surface_zone']['score'] or 0) -
            (normalized['L6_resonance_field']['score'] or 0), 4),
        'atmosphere_to_gec_gap':     round(
            (normalized['L3_atmosphere']['score'] or 0) -
            (normalized['L5_global_electric_circuit']['score'] or 0), 4),
        'external_pressure_low':     bool(
            (normalized['L0_external_drivers']['score'] or 1) < 0.3),
        'resonance_confirmed':        bool(
            (normalized['L6_resonance_field']['score'] or 0) > 0.4 and
            (normalized['L3_atmosphere']['score'] or 0) > 0.4),
        'transition_state':           bool(
            (normalized['L2_surface_zone']['score'] or 0) > 0.4 and
            (normalized['L3_atmosphere']['score'] or 0) < 0.35),
        'interpretation': (
            'surface_prepared_but_atmosphere_not_activated'
            if (normalized['L2_surface_zone']['score'] or 0) > 0.4 and
               (normalized['L3_atmosphere']['score'] or 0) < 0.35
            else 'coupled_activation' if
               (normalized['L3_atmosphere']['score'] or 0) > 0.4 and
               (normalized['L6_resonance_field']['score'] or 0) > 0.4
            else 'background_state'
        ),
    })(  ),

    # Kp context (different sources/time windows)
    'kp_context': {
        'L0_Kp_used':    (
            normalized['L0_external_drivers']['key_metrics'].get('Kp')),
        'L4_Kp_current': (
            normalized['L4_ionosphere']['key_metrics'].get('Kp')),
        'note': (
            'Different Kp windows/sources: '
            'L0 uses broader context (Layer-0-Score-Kp), '
            'L4 uses current 1-min ionospheric state.'
        ),
    },

    # Couplings
    'couplings': couplings,

    # Event tags
    'event_tags': event_tags,

    # Thresholds (transparency)
    'thresholds': {
        'level_thresholds': LEVEL_THRESHOLDS,
        'trend_threshold':  0.05,
        'active_layer':     0.5,
        'weak_layer':       0.3,
    },

    # Layer 7 output for Layer 8 explicitly named
    'layer8_handoff': {
        'state_summary': (
            f'System state: {system_state} | Score {state_score:.3f} | '
            f'Conf {state_confidence:.0%} | Dom: {dominant_layer} | '
            f'Tags: {len(event_tags)} | History: {len(history)+1} snapshots'
        ),
        'research_questions': [
            'Which tag combinations occur together?',
            'Which layers couple most strongly?',
            'Which system states repeat over time?',
            'Which tags appear before anomalous_resonance_state?',
        ],
    },
}

# numpy cleanup
def _to_python(obj):
    if isinstance(obj, dict):  return {k: _to_python(v) for k, v in obj.items()}
    if isinstance(obj, list):  return [_to_python(v) for v in obj]
    if isinstance(obj, np.bool_):    return bool(obj)
    if isinstance(obj, np.integer):  return int(obj)
    if isinstance(obj, np.floating): return None if np.isnan(obj) else float(obj)
    return obj
earth_field_state = _to_python(earth_field_state)

with open(STATE_FILE, 'w', encoding='utf-8') as f:
    json.dump(earth_field_state, f, indent=2, ensure_ascii=False)
print(f'✅ {STATE_FILE} saved')

# ============================================================
# HISTORY ARCHIVE – layer7_test_history.jsonl
# One line per engine run (for Layer 8)
# ============================================================

with open(HISTORY_FILE, 'a', encoding='utf-8') as f:
    f.write(json.dumps(earth_field_state, ensure_ascii=False) + '\n')
print(f'✅ Snapshot appended to {HISTORY_FILE} (archive for Layer 8)')
print(f'   Archive size: {len(history)+1} snapshots')

# Compact summary
print('\n' + '=' * 78)
print('EARTH FIELD STATE – SUMMARY')
print('=' * 78)
print(f'  Timestamp:          {RUN_TIMESTAMP}')
print(f'  System state:       {system_state}')
print(f'  State score:        {state_score:.3f}  ({avg_confidence:.0%} confidence)')
print(f'  Dominant layer:     {dominant_layer}')
print(f'  Active layers:      {active_layers}')
print(f'  Event tags:         {event_tags}')
print(f'  Strongest coupling: {max(couplings, key=lambda c: c["strength"])["from"]} → {max(couplings, key=lambda c: c["strength"])["to"]} ({max(c["strength"] for c in couplings):.2f})')
print(f'  History archive:    {len(history)+1} snapshots in {HISTORY_FILE}')
print('=' * 78)

✅ layer7_test_state.json saved
✅ Snapshot appended to layer7_test_history.jsonl (archive for Layer 8)
   Archive size: 9 snapshots

EARTH FIELD STATE – SUMMARY
  Timestamp:          2026-05-12T07:27:09.478526Z
  System state:       seasonal_transition_state
  State score:        0.359  (94% confidence)
  Dominant layer:     L2_surface_zone
  Active layers:      []
  Event tags:         ['el_nino_developing', 'elevated_seismicity', 'low_global_activity', 'non_geometric_dominance', 'state_seasonal_transition_state']
  Strongest coupling: L4_ionosphere → L6_resonance_field (0.41)
  History archive:    9 snapshots in layer7_test_history.jsonl


---
## Summary Layer 7
| Aspect | Content |
|--------|---------|
| **Role** | Earth Field State Engine – state machine + coupling model + data archive |
| **Input** | `layer{0..6}_test_state.json` |
| **Output (current)** | `layer7_test_state.json` – complete system state |
| **Output (archive)** | `layer7_test_history.jsonl` – one snapshot per engine run |
| **System states** | 8 classes (background, atmospheric_driven, space_weather, geomagnetic, mixed, cavity, seasonal, anomalous, low_confidence) |
| **Couplings** | 7 defined paths (L0→L4, L3→L5/L6, L4→L6, L5→L6, L0→L5, L2→L3) |
| **Trends** | Δ1h, Δ6h, Δ24h, volatility (from history) |
| **Event tags** | Automatically derived from layer flags + system state |
| **→ Layer 8** | Research on history, tag patterns, coupling correlations |
> **Important:** Each engine run appends a new snapshot to `layer7_test_history.jsonl`. Layer 8 will work on this growing archive.
> **Next step:** `layer8_Research.ipynb` – Research / Hypotheses / System questions